In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(
    model=model,
    fletcher=False,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.11785468459129333
epoch:  1, loss: 0.08100894838571548
epoch:  2, loss: 0.050515156239271164
epoch:  3, loss: 0.03337489441037178
epoch:  4, loss: 0.03245077654719353
epoch:  5, loss: 0.029173530638217926
epoch:  6, loss: 0.025665823370218277
epoch:  7, loss: 0.023203782737255096
epoch:  8, loss: 0.020829936489462852
epoch:  9, loss: 0.01971891149878502
epoch:  10, loss: 0.01820608787238598
epoch:  11, loss: 0.017146525904536247
epoch:  12, loss: 0.01641065999865532
epoch:  13, loss: 0.014762913808226585
epoch:  14, loss: 0.013780207373201847
epoch:  15, loss: 0.013194499537348747
epoch:  16, loss: 0.012818033806979656
epoch:  17, loss: 0.01179514266550541
epoch:  18, loss: 0.011086066253483295
epoch:  19, loss: 0.010746833868324757
epoch:  20, loss: 0.01056160032749176
epoch:  21, loss: 0.010111710987985134
epoch:  22, loss: 0.009725380688905716
epoch:  23, loss: 0.009579473175108433
epoch:  24, loss: 0.00950777530670166
epoch:  25, loss: 0.009301136247813702
epoch:

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9624872803688049
Test metrics:  R2 = 0.930679202079773


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(
    model=model,
    fletcher=True,
    solver="pinv"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0

/home/eugeniolr/Documents/Eugenio/torch_numopt/src/torch_numopt/utils.py:60: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:690.)
  U, S, Vt = torch.linalg.svd(mat)


, loss: 0.5455030202865601
epoch:  1, loss: 0.46389341354370117
epoch:  2, loss: 0.38348495960235596
epoch:  3, loss: 0.2685472369194031
epoch:  4, loss: 0.2293744534254074
epoch:  5, loss: 0.15191787481307983
epoch:  6, loss: 0.07941494882106781
epoch:  7, loss: 0.04115256294608116
epoch:  8, loss: 0.03247438371181488
epoch:  9, loss: 0.0231692586094141
epoch:  10, loss: 0.01757393404841423
epoch:  11, loss: 0.01382985059171915
epoch:  12, loss: 0.013451731763780117
epoch:  13, loss: 0.011915115639567375
epoch:  14, loss: 0.011178993619978428
epoch:  15, loss: 0.010825429111719131
epoch:  16, loss: 0.010615946725010872
epoch:  17, loss: 0.009809616953134537
epoch:  18, loss: 0.009436438791453838
epoch:  19, loss: 0.009298363700509071
epoch:  20, loss: 0.009237964637577534
epoch:  21, loss: 0.008871904574334621
epoch:  22, loss: 0.008789768442511559
epoch:  23, loss: 0.008763540536165237
epoch:  24, loss: 0.00871966127306223
epoch:  25, loss: 0.008673145435750484
epoch:  26, loss: 0.00

In [9]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6779686808586121
Test metrics:  R2 = 0.6320192813873291
